# DINO-WM Foundation

Operational notebook for DINO-WM setup, dry-runs, guarded launches, and summaries. Training, fine-tuning, precompute, and planning are delegated to repository scripts.

In [2]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = "https://github.com/Thomas-Georges/World_Models_LAS.git"
REPO_DIR = Path("/content/World_Models_LAS")
REPO_BRANCH = "main"
REPO_RUN_CWD = REPO_DIR.parent


def ensure_run_cwd() -> None:
    REPO_RUN_CWD.mkdir(parents=True, exist_ok=True)
    os.chdir(REPO_RUN_CWD)


def run_git(args, *, check=True):
    ensure_run_cwd()
    print("$", " ".join(map(str, args)))
    result = subprocess.run(
        args,
        check=False,
        cwd=str(REPO_RUN_CWD),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, args, output=result.stdout)
    return result


def clone_repo():
    ensure_run_cwd()
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run_git(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])


def sync_repo():
    ensure_run_cwd()
    if not (REPO_DIR / ".git").is_dir():
        clone_repo()
        return

    run_git(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])
    fetch = run_git(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=False)
    if fetch.returncode != 0:
        print(f"Fetch failed for existing checkout at {REPO_DIR}; replacing it with a fresh clone.")
        clone_repo()
        return

    run_git(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
    run_git(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"])


sync_repo()


command = ["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"]
print("$", " ".join(command))
head = subprocess.check_output(command, text=True, cwd=str(REPO_RUN_CWD)).strip()
print("Repository HEAD:", head)

$ git clone --branch main --single-branch https://github.com/Thomas-Georges/World_Models_LAS.git /content/World_Models_LAS
Cloning into '/content/World_Models_LAS'...
$ git -C /content/World_Models_LAS log --oneline -1
Repository HEAD: dc53d2e Adding a safeguard for torch-torchvision mismatch


In [3]:
from pathlib import Path
import os

DRIVE_MOUNT = Path("/content/drive")
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()

if IN_COLAB:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT), force_remount=False)
    if not (DRIVE_MOUNT / "MyDrive").exists():
        raise RuntimeError("Google Drive mount did not expose /content/drive/MyDrive.")
    print(f"Google Drive mounted at {DRIVE_MOUNT}")
else:
    print("Not running in Colab; skipping Google Drive mount.")

Mounted at /content/drive
Google Drive mounted at /content/drive


In [4]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

REPO = REPO_DIR if 'REPO_DIR' in globals() and (REPO_DIR / '.git').is_dir() else Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

DRIVE_ROOT = Path(os.environ.get('WM_POC_DRIVE_ROOT', '/content/drive/MyDrive/wm_poc'))
os.environ.setdefault('DINO_WM_REPO', str(DRIVE_ROOT / 'external_repos/dino_wm'))

CONFIG_SMOKE = Path('configs/dino_wm/smoke_pointmaze.yaml')
CONFIG_SMOKE_LATENT = Path('configs/dino_wm/smoke_pointmaze_latent.yaml')
CONFIG_POINTMAZE = Path('configs/dino_wm/pointmaze_scratch_a100.yaml')
CONFIG_OOM_SAFE = Path('configs/dino_wm/pointmaze_oom_safe.yaml')
CONFIG_FULL_NODECODER_BF16 = Path('configs/dino_wm/pointmaze_full_nodecoder_bf16.yaml')
CONFIG_FULL_NODECODER_T4 = Path('configs/dino_wm/pointmaze_full_nodecoder_t4.yaml')
CONFIG_LOW_SCRATCH = Path('configs/dino_wm/pointmaze_lowdata_scratch_a100.yaml')
CONFIG_LOW_FT = Path('configs/dino_wm/pointmaze_lowdata_finetune_a100.yaml')
SUMMARY_ROOT = Path(os.environ.get('DINO_LOG_ROOT', str(DRIVE_ROOT / 'logs/dino_wm')))
CKPT_ROOT = Path(os.environ.get('DINO_CKPT_ROOT', str(DRIVE_ROOT / 'checkpoints/dino_wm')))
DINO_MONITOR_INTERVAL = float(os.environ.get('DINO_MONITOR_INTERVAL', '15'))
print(f'Repo: {REPO}')
print(f'DINO-WM upstream repo: {os.environ["DINO_WM_REPO"]}')
print(f'Summary root: {SUMMARY_ROOT}')
print(f'Checkpoint root: {CKPT_ROOT}')
print(f'DINO monitor interval: {DINO_MONITOR_INTERVAL:g}s')

Repo: /content/World_Models_LAS
DINO-WM upstream repo: /content/drive/MyDrive/wm_poc/external_repos/dino_wm
Summary root: /content/drive/MyDrive/wm_poc/logs/dino_wm
Checkpoint root: /content/drive/MyDrive/wm_poc/checkpoints/dino_wm
DINO monitor interval: 15s


In [5]:
from wm_poc.dino_wm.notebook_monitor import run_dino_with_live_display


def show_command(cmd, env=None):
    prefix = ''
    if env:
        prefix = ' '.join(f'{key}={shlex.quote(str(value))}' for key, value in env.items()) + ' '
    print('$ ' + prefix + shlex.join([str(part) for part in cmd]))


def run_if_safe(cmd, *, env=None, check=False):
    show_command(cmd, env=env)
    return subprocess.run([str(part) for part in cmd], env={**os.environ, **(env or {})}, check=check)


def run_guarded(cmd):
    if os.environ.get('RUN_DINO_WM') != '1':
        show_command(cmd)
        print("Skipped. Run os.environ['RUN_DINO_WM'] = '1' and rerun this cell to execute.")
        return None
    env = {**os.environ, 'RUN_DINO_WM': '1'}
    show_command(cmd, env={'RUN_DINO_WM': '1'})
    result = subprocess.run(
        [str(part) for part in cmd],
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {shlex.join([str(part) for part in cmd])}")
    return result


def enable_dino_runs(monitor_interval=None):
    os.environ['RUN_DINO_WM'] = '1'
    if monitor_interval is not None:
        os.environ['DINO_MONITOR_INTERVAL'] = str(monitor_interval)
        globals()['DINO_MONITOR_INTERVAL'] = float(monitor_interval)
    print(f"RUN_DINO_WM={os.environ['RUN_DINO_WM']}")
    print(f"DINO monitor interval: {globals().get('DINO_MONITOR_INTERVAL', os.environ.get('DINO_MONITOR_INTERVAL', 15))}s")


def run_dino_live(stage, **kwargs):
    if os.environ.get('RUN_DINO_WM') != '1':
        print("DINO-WM execution is disabled for safety.")
        print("Run enable_dino_runs(monitor_interval=5) and rerun this cell to launch with live monitoring.")
        return None
    kwargs.setdefault('repo_dir', REPO)
    kwargs.setdefault('monitor_interval', DINO_MONITOR_INTERVAL)
    return run_dino_with_live_display(stage, **kwargs)


## Environment and Drive Setup

In [6]:
run_if_safe(['python', 'scripts/verify_environment.py', '--cpu-only'])
run_if_safe(['python', 'scripts/verify_drive_layout.py', '--dry-run'])

$ python scripts/verify_environment.py --cpu-only
$ python scripts/verify_drive_layout.py --dry-run


CompletedProcess(args=['python', 'scripts/verify_drive_layout.py', '--dry-run'], returncode=0)

## Upstream DINO-WM Setup or Verification

In [7]:
run_if_safe(['python', 'scripts/dino_wm/verify_dino_wm_env.py', '--allow-cpu', '--allow-missing-upstream', '--skip-dependency-check'])

upstream_repo = Path(os.environ.get('DINO_WM_REPO', str(DRIVE_ROOT / 'external_repos/dino_wm')))
if not upstream_repo.is_absolute():
    upstream_repo = REPO / upstream_repo
missing_upstream = not (upstream_repo / 'train.py').is_file() or not (upstream_repo / 'plan.py').is_file()

if missing_upstream:
    print(f'Upstream DINO-WM is missing at {upstream_repo}; cloning it into Drive.')
    run_if_safe(
        ['bash', 'scripts/dino_wm/setup_dino_wm.sh'],
        env={'DINO_WM_REPO': str(upstream_repo), 'DINO_INSTALL_DEPS': '0'},
        check=True,
    )
else:
    print(f'Upstream DINO-WM found: {upstream_repo}')

print('DINO-WM Python dependency install is deferred to the guarded smoke/train cells.')

$ python scripts/dino_wm/verify_dino_wm_env.py --allow-cpu --allow-missing-upstream --skip-dependency-check
Upstream DINO-WM found: /content/drive/MyDrive/wm_poc/external_repos/dino_wm
DINO-WM Python dependency install is deferred to the guarded smoke/train cells.


## Dataset Verification

In [8]:
data_root = Path(os.environ.get('DINO_WM_DATA_ROOT', '/content/drive/MyDrive/wm_poc/data/dino_wm'))
verify_cmd = ['python', 'scripts/dino_wm/verify_data.py', '--config', str(CONFIG_SMOKE)]
download_cmd = ['python', 'scripts/dino_wm/download_data.py', '--dataset', 'point_maze', '--data-root', str(data_root)]

print('Checking official DINO-WM dataset archive metadata.')
run_if_safe(download_cmd + ['--list'])

verify_result = run_if_safe(verify_cmd)
if verify_result.returncode != 0:
    print(f'PointMaze data is missing or invalid under {data_root}; downloading official DINO-WM PointMaze data.')
    run_if_safe(download_cmd + ['--execute'], check=True)
    run_if_safe(verify_cmd, check=True)

Checking official DINO-WM dataset archive metadata.
$ python scripts/dino_wm/download_data.py --dataset point_maze --data-root /content/drive/MyDrive/wm_poc/data/dino_wm --list
$ python scripts/dino_wm/verify_data.py --config configs/dino_wm/smoke_pointmaze.yaml


## Feature Cache Verification and Dry-Run

In [9]:
run_if_safe(['python', 'scripts/dino_wm/precompute_latents.py', '--config', str(CONFIG_SMOKE), '--dry-run'])

$ python scripts/dino_wm/precompute_latents.py --config configs/dino_wm/smoke_pointmaze.yaml --dry-run


CompletedProcess(args=['python', 'scripts/dino_wm/precompute_latents.py', '--config', 'configs/dino_wm/smoke_pointmaze.yaml', '--dry-run'], returncode=0)

## Command Dry-Runs

In [10]:
for config in [
    CONFIG_SMOKE,
    CONFIG_SMOKE_LATENT,
    CONFIG_OOM_SAFE,
    CONFIG_FULL_NODECODER_BF16,
    CONFIG_FULL_NODECODER_T4,
    CONFIG_POINTMAZE,
    CONFIG_LOW_SCRATCH,
    CONFIG_LOW_FT,
]:
    run_if_safe(['python', 'scripts/dino_wm/build_commands.py', '--config', str(config), '--stage', 'train', '--print'])

$ python scripts/dino_wm/build_commands.py --config configs/dino_wm/smoke_pointmaze.yaml --stage train --print
$ python scripts/dino_wm/build_commands.py --config configs/dino_wm/smoke_pointmaze_latent.yaml --stage train --print
$ python scripts/dino_wm/build_commands.py --config configs/dino_wm/pointmaze_oom_safe.yaml --stage train --print
$ python scripts/dino_wm/build_commands.py --config configs/dino_wm/pointmaze_full_nodecoder_bf16.yaml --stage train --print
$ python scripts/dino_wm/build_commands.py --config configs/dino_wm/pointmaze_full_nodecoder_t4.yaml --stage train --print
$ python scripts/dino_wm/build_commands.py --config configs/dino_wm/pointmaze_scratch_a100.yaml --stage train --print
$ python scripts/dino_wm/build_commands.py --config configs/dino_wm/pointmaze_lowdata_scratch_a100.yaml --stage train --print
$ python scripts/dino_wm/build_commands.py --config configs/dino_wm/pointmaze_lowdata_finetune_a100.yaml --stage train --print


## Run Profile and Preflight

One place for the launch environment used by every guarded run below: GPU detection picks bf16 vs fp16 and the stride-1 vs stride-2 full-run config, the latent cache is pinned to local disk, stale-process cleanup is armed, and free disk space is checked against the ~30 GB the full latent cache needs.

In [11]:
import shutil
import torch

# Environment shared by every guarded launch below (smoke, full, fine-tune).
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_MODE"] = "offline"
# Kill leftover DINO-WM workers from interrupted cells at every launch.
os.environ["DINO_KILL_STALE"] = "1"
# Latent cache stays on fast local disk; Drive FUSE random reads would
# re-create the old I/O bottleneck (the scripts warn if this points at Drive).
os.environ["DINO_WM_FEATURE_CACHE"] = "/content/wm_poc_latent_cache"

# bf16 needs Ampere or newer (A100/L4); T4 only emulates it, so use fp16 there.
BF16_OK = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
os.environ["DINO_MIXED_PRECISION"] = "bf16" if BF16_OK else "fp16"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"

# Full-run profile. A100/L4 keep the paper's exact stride-1 window
# distribution; T4 falls back to the stride-2 throughput profile
# (~30 min/epoch instead of ~60). Override CONFIG_FULL by hand to force
# stride-1 windows on T4.
CONFIG_FULL = CONFIG_FULL_NODECODER_BF16 if BF16_OK else CONFIG_FULL_NODECODER_T4

# Checkpoint cadence (disconnect insurance for paid GPU time):
#  - SAVE_EVERY_EPOCHS: full epoch checkpoints (model_latest.pth + model_N.pth).
#  - SAVE_EVERY_STEPS: rolling intra-epoch state_dict checkpoint
#    (model_latest_step.pth) every N optimizer steps; relaunching the same
#    cell resumes mid-epoch from it. 0 disables and removes the step patch.
# Defaults bound a crash to a few minutes of lost work on either GPU.
SAVE_EVERY_EPOCHS = 1
SAVE_EVERY_STEPS = 2000 if BF16_OK else 500
os.environ["DINO_SAVE_EVERY_EPOCHS"] = str(SAVE_EVERY_EPOCHS)
if SAVE_EVERY_STEPS > 0:
    os.environ["DINO_SAVE_EVERY_STEPS"] = str(SAVE_EVERY_STEPS)
    os.environ["DINO_PATCH_STEP_CHECKPOINTING"] = "1"
else:
    os.environ.pop("DINO_SAVE_EVERY_STEPS", None)
    os.environ["DINO_PATCH_STEP_CHECKPOINTING"] = "0"

free_gb = shutil.disk_usage("/content").free / 1e9 if Path("/content").exists() else float("nan")
print(f"GPU: {GPU_NAME} | precision: {os.environ['DINO_MIXED_PRECISION']} | vCPUs: {os.cpu_count()}")
print(f"Full-run config: {CONFIG_FULL}")
print(f"Checkpoints: every {SAVE_EVERY_EPOCHS} epoch(s) + rolling every {SAVE_EVERY_STEPS} step(s)")
print(f"Latent cache: {os.environ['DINO_WM_FEATURE_CACHE']} (local disk free: {free_gb:.0f} GB; full cache needs ~30 GB)")
if free_gb < 40:
    print("WARNING: local disk is tight for the full latent cache; free space or expect precompute to fail.")

GPU: NVIDIA A100-SXM4-40GB | precision: bf16 | vCPUs: 12
Full-run config: configs/dino_wm/pointmaze_full_nodecoder_bf16.yaml
Checkpoints: every 1 epoch(s) + rolling every 2000 step(s)
Latent cache: /content/wm_poc_latent_cache (local disk free: 204 GB; full cache needs ~30 GB)


## Smoke Run (latent-path crash test)

Run this before any long training. It exercises the **exact pipeline the full run uses**, shrunk to 20 rollouts and ~10-15 minutes: dependency install, upstream patches (mixed precision + latent bypass), DINO hub download, latent precompute with manifest, no-decoder latent training for one epoch with dataloader workers, checkpoint + summary writing, and a tiny CEM planning eval that loads the checkpoint and drives the **real** encoder on environment observations.

If this cell completes with `state: finished`, every moving part of the full run and the planning path has been crash-tested under real Colab conditions. The 20 episodes it encodes are reused by the full run's cache (precompute resumes, it does not start over).

In [13]:
# Dry-run print of what the smoke will execute:
run_if_safe(["bash", "scripts/dino_wm/run_smoke.sh"], env={"RUN_DINO_WM": "0", "DINO_CONFIG": str(CONFIG_SMOKE_LATENT)})

enable_dino_runs(monitor_interval=10)
run_dino_live("smoke", config=CONFIG_SMOKE_LATENT)

# Legacy online-encoding smoke (image path, no latent cache), kept for
# diagnosing the encoder/image pipeline itself:
# run_dino_live("smoke", config=CONFIG_SMOKE)

$ RUN_DINO_WM=0 DINO_CONFIG=configs/dino_wm/smoke_pointmaze_latent.yaml bash scripts/dino_wm/run_smoke.sh
RUN_DINO_WM=1
DINO monitor interval: 10.0s


0

In [ ]:
# Manual log inspection for a run dir (works for any run; smoke shown here).
# from pathlib import Path
# import time

# run_dir = SUMMARY_ROOT / "smoke_pointmaze_latent_seed0"

# for name in ["launcher.log", "stdout.log", "stderr.log", "status.json", "planning/status_cem.json"]:
#     p = run_dir / name
#     print("\n==", p, "==")
#     if not p.exists():
#         print("missing")
#         continue
#     print("size:", p.stat().st_size)
#     print("updated seconds ago:", round(time.time() - p.stat().st_mtime, 1))
#     print("\n".join(p.read_text(errors="replace").splitlines()[-120:]))

## PointMaze OOM-Safe Run

In [ ]:
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# os.environ["DINO_PATCH_STEP_CHECKPOINTING"] = "0"
# os.environ.pop("DINO_SAVE_EVERY_STEPS", None)
# os.environ["DINO_SAVE_EVERY_EPOCHS"] = "5"
# os.environ["DINO_MIXED_PRECISION"] = "bf16"
# os.environ["WANDB_MODE"] = "offline"
# os.environ["DINO_MAX_WALL_MINUTES"] = "120"

# SAFE_CONFIG = CONFIG_OOM_SAFE

# run_if_safe(
#     ["bash", "scripts/dino_wm/run_experiment.sh", "--config", str(SAFE_CONFIG), "--skip-cache", "--skip-plan"],
#     env={"RUN_DINO_WM": "0"},
# )

# enable_dino_runs(monitor_interval=5)
# run_dino_live("experiment", config=SAFE_CONFIG, skip_cache=True, skip_plan=True)


## PointMaze Full No-Decoder Run (cached latents)

Trains the predictor on **precomputed DINO latents**: `run_experiment.sh` installs the latent-cache support files, precomputes frozen DINO patch latents once per episode (resumable), then training reads 4-frame latent slices from memory-mapped local files.

**The cell is live and self-gating**: it skips entirely when any full-run checkpoint is already complete (no GPU time spent — that checkpoint is the fine-tune's source), resumes a partial run from its rolling/epoch checkpoints, and only trains from zero when nothing exists. The config comes from the preflight cell (stride-1 bf16 on A100/L4, stride-2 fp16 profile on T4).

In [14]:
# Full no-decoder training - self-gating: skips when ANY full-run checkpoint
# is complete (it is the fine-tune's source and needs no retraining), resumes
# a partial run from its checkpoints, and only trains from zero when nothing
# exists yet.
from wm_poc.dino_wm.configs import load_config, resolve_config
from wm_poc.dino_wm.resume import latest_epoch_checkpoint, training_complete

completed_full = None
for candidate in (CONFIG_FULL, CONFIG_FULL_NODECODER_BF16, CONFIG_FULL_NODECODER_T4):
    cfg = resolve_config(load_config(candidate))
    if training_complete(cfg):
        completed_full = (candidate, cfg["run_name"], latest_epoch_checkpoint(cfg))
        break

if completed_full is not None:
    print(f"Full run already complete: {completed_full[1]} (epoch {completed_full[2]}); skipping training.")
    print("Its checkpoint is the fine-tune source. Set DINO_FORCE_RESTART=1 manually to redo it.")
else:
    cfg = resolve_config(load_config(CONFIG_FULL))
    done = latest_epoch_checkpoint(cfg)
    if done:
        print(f"Resuming {cfg['run_name']} from epoch {done}.")
    os.environ["DINO_MAX_WALL_MINUTES"] = "600"  # raised from 220/400 for reruns (full run)
    print(f"Full run: {CONFIG_FULL} | GPU: {GPU_NAME} | precision: {os.environ['DINO_MIXED_PRECISION']}")
    enable_dino_runs(monitor_interval=15)
    run_dino_live("experiment", config=CONFIG_FULL, skip_cache=False, skip_plan=True)

Full run already complete: pointmaze_full_nodecoder_t4_fp16_b32_stride2_seed0 (epoch 10); skipping training.
Its checkpoint is the fine-tune source. Set DINO_FORCE_RESTART=1 manually to redo it.


## PointMaze Low-Data Scratch and Fine-Tune

Transfer comparison on 300+60 rollouts. All cells are **live and self-gating** — running the notebook top to bottom does only the outstanding work:

- **Scratch baseline**: skips its completed training, but **tops up the planning evaluation to `n_evals=200`** (plan-only launch) if the existing eval was the old 50-eval one. Nothing runs once both are done.
- **Fine-tune**: always a **fresh restart** (`DINO_FORCE_RESTART=1` scoped to the launch moves the previous fine-tune's checkpoints aside), retrains the 20 epochs from the full run's checkpoint, and ends with 200-eval CEM planning.

Both train predictor-only on cached latents, matching the full run's training shape; the command builder rejects any decoder+latents combination outright.

In [15]:
# Low-data scratch baseline - self-gating: skips completed training, resumes
# partial training, and tops up the planning evaluation to the config's
# n_evals=200 when the existing eval is older/smaller.
from wm_poc.dino_wm.checkpoints import find_latest_checkpoint
from wm_poc.dino_wm.configs import load_config, resolve_config
from wm_poc.dino_wm.resume import latest_epoch_checkpoint, planning_complete, training_complete

scratch_cfg = resolve_config(load_config(CONFIG_LOW_SCRATCH))
if not training_complete(scratch_cfg):
    done = latest_epoch_checkpoint(scratch_cfg)
    print(f"Scratch training {'resumes from epoch ' + str(done) if done else 'starts fresh'}.")
    os.environ["DINO_MAX_WALL_MINUTES"] = "80" if BF16_OK else "400"
    enable_dino_runs(monitor_interval=15)
    run_dino_live("experiment", config=CONFIG_LOW_SCRATCH)
elif not planning_complete(scratch_cfg):
    ckpt = find_latest_checkpoint(CKPT_ROOT / "outputs" / str(scratch_cfg["run_name"]))
    print(f"Scratch training complete; re-running planning at n_evals=200 from {ckpt}")
    os.environ["DINO_MAX_WALL_MINUTES"] = "120" if BF16_OK else "360"
    enable_dino_runs(monitor_interval=15)
    run_dino_live("plan", config=CONFIG_LOW_SCRATCH, checkpoint=str(ckpt))
else:
    print("Scratch training and 200-eval planning both complete; nothing to do.")

Scratch training and 200-eval planning both complete; nothing to do.


In [16]:
# Fresh low-data fine-tune from the full run's checkpoint, ending with the
# 200-eval CEM planning (from the config). DINO_FORCE_RESTART moves the
# previous fine-tune's checkpoints aside so this trains from the source
# checkpoint again instead of resuming the old fine-tune.
from wm_poc.dino_wm.checkpoints import find_latest_checkpoint
from wm_poc.dino_wm.configs import load_config, resolve_config


def resolve_source_checkpoint():
    explicit = os.environ.get("DINO_POINTMAZE_SOURCE_CKPT")
    if explicit:
        if Path(explicit).expanduser().is_file():
            return explicit, "DINO_POINTMAZE_SOURCE_CKPT"
        print(f"DINO_POINTMAZE_SOURCE_CKPT does not exist, ignoring: {explicit}")
    for candidate in (CONFIG_FULL, CONFIG_FULL_NODECODER_BF16, CONFIG_FULL_NODECODER_T4, CONFIG_POINTMAZE):
        run_name = str(resolve_config(load_config(candidate)).get("run_name"))
        ckpt = find_latest_checkpoint(CKPT_ROOT / "outputs" / run_name)
        if ckpt is not None:
            return str(ckpt), f"latest checkpoint of {run_name}"
    return None, None


source_ckpt, source_desc = resolve_source_checkpoint()
if source_ckpt is None:
    print("No source checkpoint found under", CKPT_ROOT / "outputs")
    print("Run the full no-decoder training first, or set DINO_POINTMAZE_SOURCE_CKPT to a model_latest.pth path.")
else:
    os.environ["DINO_POINTMAZE_SOURCE_CKPT"] = source_ckpt
    print(f"Fine-tune source ({source_desc}): {source_ckpt}")
    # 20 fine-tune epochs + 200-eval planning: ~1 h on A100, ~5-6 h on T4.
    os.environ["DINO_MAX_WALL_MINUTES"] = "80" if BF16_OK else "400"
    os.environ["DINO_FORCE_RESTART"] = "1"
    try:
        enable_dino_runs(monitor_interval=15)
        run_dino_live("experiment", config=CONFIG_LOW_FT)
    finally:
        os.environ.pop("DINO_FORCE_RESTART", None)

Fine-tune source (latest checkpoint of pointmaze_full_nodecoder_t4_fp16_b32_stride2_seed0): /content/drive/MyDrive/wm_poc/checkpoints/dino_wm/outputs/pointmaze_full_nodecoder_t4_fp16_b32_stride2_seed0/checkpoints/model_latest.pth
RUN_DINO_WM=1
DINO monitor interval: 15.0s


In [ ]:
%%bash
RUN_DINO_WM=1 bash scripts/dino_wm/run_plan.sh --config configs/dino_wm/pointmaze_lowdata_scratch_a100.yaml
RUN_DINO_WM=1 bash scripts/dino_wm/run_plan.sh --config configs/dino_wm/pointmaze_lowdata_finetune_a100.yaml

## Planning Evaluation (full checkpoint, n_evals=200)

Standalone CEM planning of the **full run's checkpoint** at the same 200-eval budget as the fine-tune, so the three-way comparison shares one statistical footing. The cell auto-resolves the latest full-run checkpoint (or honors `DINO_POINTMAZE_SOURCE_CKPT`), and the command builder derives `model_name`/`model_epoch` from the checkpoint path. Rolling `model_latest_step.pth` files are rejected — plan from epoch checkpoints.

A latent-trained checkpoint's saved config references the latent cache on session-local disk, so `run_plan.sh` rebuilds any missing cache coverage before planning (resumable; a no-op when covered). With the evaluator video patch, this also regenerates per-episode success/failure videos for the decoder-free checkpoints.

In [17]:
# Standalone planning of the full checkpoint at n_evals=200 - self-gating:
# skips when a completed 200-eval evaluation already exists for that run.
import json as _json
from wm_poc.dino_wm.checkpoints import find_latest_checkpoint
from wm_poc.dino_wm.configs import load_config, resolve_config
from wm_poc.dino_wm.resume import planning_complete

checkpoint = os.environ.get("DINO_POINTMAZE_SOURCE_CKPT")
if not checkpoint:
    for candidate in (CONFIG_FULL, CONFIG_FULL_NODECODER_BF16, CONFIG_FULL_NODECODER_T4):
        run_name = str(resolve_config(load_config(candidate)).get("run_name"))
        ckpt = find_latest_checkpoint(CKPT_ROOT / "outputs" / run_name)
        if ckpt is not None:
            checkpoint = str(ckpt)
            break

if not checkpoint:
    print("No checkpoint found under", CKPT_ROOT / "outputs")
    print("Run the full no-decoder training first, or set DINO_POINTMAZE_SOURCE_CKPT to a model_latest.pth path.")
else:
    # Attribute planning artifacts to the run that produced the checkpoint.
    ckpt_run = Path(checkpoint).parent.parent.name
    plan_config = CONFIG_FULL_NODECODER_T4 if "_t4_" in ckpt_run else CONFIG_FULL_NODECODER_BF16
    plan_cfg = resolve_config(load_config(plan_config))
    if planning_complete(plan_cfg):
        print(f"200-eval planning already complete for {ckpt_run}; nothing to do.")
    else:
        print(f"Planning from checkpoint: {checkpoint}")
        # 200-eval CEM takes ~20-30 min on A100 but several hours on T4; without
        # this the plan command would fall back to the config's 60-min wall.
        os.environ["DINO_MAX_WALL_MINUTES"] = "120" if BF16_OK else "360"
        enable_dino_runs(monitor_interval=15)
        run_dino_live("plan", config=plan_config, checkpoint=checkpoint)

200-eval planning already complete for pointmaze_full_nodecoder_t4_fp16_b32_stride2_seed0; nothing to do.


## Summary Table and Artifact Links

In [18]:
summary_dir = SUMMARY_ROOT / '_summary'
run_if_safe(['python', 'scripts/dino_wm/summarize_runs.py', '--root', str(SUMMARY_ROOT), '--out', str(summary_dir)])
summary_csv = summary_dir / 'summary.csv'
if summary_csv.exists():
    try:
        import pandas as pd
        display(pd.read_csv(summary_csv))
    except Exception:
        print(summary_csv.read_text())
else:
    print(f'No summary yet: {summary_csv}')
print(f'Artifacts root: {SUMMARY_ROOT}')

$ python scripts/dino_wm/summarize_runs.py --root /content/drive/MyDrive/wm_poc/logs/dino_wm --out /content/drive/MyDrive/wm_poc/logs/dino_wm/_summary
run_name,config_name,env,seed,mode,source_checkpoint,max_train_trajectories,max_val_trajectories,completed,timed_out,failed,train_wall_minutes,plan_wall_minutes,best_epoch,final_val_loss_pred_hstep,best_success_rate,final_goal_latent_distance,checkpoint_path,run_dir
pointmaze_full_nodecoder_t4_fp16_b32_stride2_seed0,config.yaml,point_maze,0,scratch,,2000,200,True,False,False,10.816666666666666,47.3,,,0.23,,,/content/drive/MyDrive/wm_poc/logs/dino_wm/pointmaze_full_nodecoder_t4_fp16_b32_stride2_seed0
pointmaze_lowdata_finetune_a100_seed0,config.yaml,point_maze,0,finetune,/content/drive/MyDrive/wm_poc/checkpoints/dino_wm/outputs/pointmaze_full_nodecoder_t4_fp16_b32_stride2_seed0/checkpoints/model_latest.pth,300,60,True,False,False,47.766666666666666,47.3,20,0.0123,0.22,,,/content/drive/MyDrive/wm_poc/logs/dino_wm/pointmaze_lowdata_finetune